# Premier League fixtures - 2026 season

Fetches the complete fixture response from API-Football and saves the untransformed API response as a timestamped JSON snapshot in `data/bronze/fixtures/league_id=39/season=2026/`. This endpoint currently returns all 380 fixtures in one response; adding a `page` parameter causes an API error.

Before running, make `API_FOOTBALL_KEY` available to the notebook process. A key set in your computer's environment is available when running locally; a Databricks job needs the key configured in its own environment. Run with the repository as the working directory (or from a subdirectory of it).

In [ ]:
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

import requests

API_URL = "https://v3.football.api-sports.io/fixtures"
LEAGUE_ID = 39  # Premier League
SEASON = 2026

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / ".git").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError(
        "Could not find the repository root. Start the notebook with the Git repository as the working directory."
    )

bronze_root = (
    repo_root
    / "data"
    / "bronze"
    / "fixtures"
    / f"league_id={LEAGUE_ID}"
    / f"season={SEASON}"
)
print(f"Bronze output directory: {bronze_root}")

In [ ]:
api_key = os.environ.get("API_FOOTBALL_KEY")
if not api_key:
    raise RuntimeError(
        "API_FOOTBALL_KEY is not set in this notebook process. Configure it in the environment and rerun."
    )

headers = {"x-apisports-key": api_key}
response = requests.get(
    API_URL,
    headers=headers,
    params={"league": LEAGUE_ID, "season": SEASON},
    timeout=60,
)
response.raise_for_status()
payload = response.json()

if payload.get("errors"):
    raise RuntimeError(f"API-Football returned errors: {payload['errors']}")
if not isinstance(payload.get("response"), list):
    raise ValueError("API-Football response is missing its fixture list.")

paging = payload.get("paging")
if not isinstance(paging, dict):
    raise ValueError("API-Football response is missing pagination metadata.")
if paging.get("current") != 1 or paging.get("total") != 1:
    raise ValueError(
        "Expected a single complete fixtures response. Review the API paging metadata before ingesting."
    )

pages = [payload]
fixture_count = len(payload["response"])
print(f"Fetched {fixture_count} fixtures in the complete API response.")

In [ ]:
bronze_root.mkdir(parents=True, exist_ok=True)
ingested_at = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
snapshot_dir = bronze_root / f"ingested_at={ingested_at}"
staging_dir = bronze_root / f".tmp-{ingested_at}"
staging_dir.mkdir()

try:
    for page_number, payload in enumerate(pages, start=1):
        page_path = staging_dir / f"page_{page_number}.json"
        with page_path.open("w", encoding="utf-8") as output_file:
            json.dump(payload, output_file, ensure_ascii=False, indent=2)
            output_file.write("\n")
    staging_dir.rename(snapshot_dir)
finally:
    if staging_dir.exists():
        shutil.rmtree(staging_dir)

print(f"Saved {len(pages)} raw API page(s) to {snapshot_dir}")